
# Phase 1


# Part 1: Setup and Configuration

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
raw_dir       = Path("D:/music_data/raw")
metadata_dir  = raw_dir / "fma_metadata"
processed_dir = Path("D:/music_data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Verify
print("Metadata dir exists:", metadata_dir.exists())
print("Files found:")
for f in metadata_dir.glob("*"):
    print(" ", f.name)

Metadata dir exists: True
Files found:
  checksums
  echonest.csv
  features.csv
  genres.csv
  not_found.pickle
  raw_albums.csv
  raw_artists.csv
  raw_echonest.csv
  raw_genres.csv
  raw_tracks.csv
  README.txt
  tracks.csv


In [8]:
#run and load the tracks 

tracks = pd.read_csv(metadata_dir / "tracks.csv",index_col=0, header=[0,1])

#(want to see all 106,574 tracks and 4 column groups) 
print("shape:", tracks.shape)
print("Columns (top level):", tracks.columns.get_level_values(0).unique().tolist())

shape: (106574, 52)
Columns (top level): ['album', 'artist', 'set', 'track']


In [9]:
#check # of audio files
audio_dir = Path("D:/music_data/raw/fma_large")

#list all mp3 files
mp3_files = list(audio_dir.glob("**/*.mp3"))
print(f"Total mp3 files found: {len(mp3_files)}")

# extract track ids from filenames
audio_ids = set(int(f.stem) for f in mp3_files)
metadata_ids = set(df['track_id'].values)

#sanity check
matched = audio_ids & metadata_ids
missing_audio = metadata_ids - audio_ids

print(f"Tracks in metadata: {len(metadata_ids)}")
print(f"Audio files found: {len(audio_ids)}")
print(f"Matched: {len(matched)}")
print(f"Missing audio: {len(missing_audio)}")
print(f"Coverage: {len(matched)/len(metadata_ids)*100:.1f}%")



Total mp3 files found: 106574
Tracks in metadata: 106574
Audio files found: 106574
Matched: 106574
Missing audio: 0
Coverage: 100.0%


In [10]:
#flatten and extract columns
df = pd.DataFrame()
df['track_id'] = tracks.index
df['title'] = tracks['track']['title'].values # fma has 2-row headers meaning columns are nested
df['artist'] = tracks['artist']['name'].values
df['genre'] = tracks['track']['genre_top'].values
df['duration'] = tracks['track']['duration'].values
df['subset'] = tracks['set']['subset'].values

df = df[df['track_id'].isin(audio_ids)].reset_index(drop=True)
print(f"Large subset: {len(df)} tracks")
print(f"Null genres: {df['genre'].isna().sum()}")
print(f"\nSubset breakdown:")
print(df['subset'].value_counts())
print(f"\nGenre breakdown:")
print(df['genre'].value_counts())

Large subset: 106574 tracks
Null genres: 56976

Subset breakdown:
subset
large     81574
medium    17000
small      8000
Name: count, dtype: int64

Genre breakdown:
genre
Rock                   14182
Experimental           10608
Electronic              9372
Hip-Hop                 3552
Folk                    2803
Pop                     2332
Instrumental            2079
International           1389
Classical               1230
Jazz                     571
Old-Time / Historic      554
Spoken                   423
Country                  194
Soul-RnB                 175
Blues                    110
Easy Listening            24
Name: count, dtype: int64


In [11]:
# save cleaned metadata 
out_path = processed_dir / "tracks_clean.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved {len(df)} tracks to {out_path}")



Saved 106574 tracks to D:\music_data\processed\tracks_clean.parquet
